# Notebook 3: Ringkasan Stok dengan Generative AI (RQ3)

**Pertanyaan yang dijawab (RQ3):** bagaimana Generative AI menghasilkan ringkasan kondisi stok untuk mengidentifikasi bahan yang membutuhkan perhatian, dengan ringkasan yang **hanya memuat informasi dari hasil klasifikasi**, tanpa tambahan data yang tidak ada?

**Input:** model dari Notebook 2 (`usage_model_gb_q07.joblib`) dan `dataset_model.csv`.
**Output:** laporan final per tanggal di `outputs/app_data/` yang dibaca oleh aplikasi Streamlit.

### Prinsip desain: LLM menulis narasi, kode menjaga kebenaran angka

LLM pandai merangkai kalimat, tetapi bisa salah menyalin angka, salah memilah bahan, atau menambahkan informasi yang tidak ada. Karena itu, **semua keputusan yang harus pasti dibuat oleh kode**, dan LLM hanya diberi tugas yang memang membutuhkan bahasa.

```
Cell 1   ML Prediction                 model memperkirakan pemakaian 7 hari
Cell 2   Inventory Snapshot            kondisi stok 10 bahan pada satu tanggal
Cell 2b  Rule-Based Classification     KODE menentukan bagian laporan, angka, dan satuan
Cell 5   GenAI Explanation             LLM menyalin daftar dan menulis pembuka serta saran
Cell 6   Deterministic Validation      KODE memeriksa setiap baris, angka, satuan, dan saran
Cell 6b  Final Inventory Report        lolos → teks LLM, gagal → ulang sekali, masih gagal → template
Cell 7   Evaluasi + observability      30 tanggal acak, latency, error rate, token
Cell 8   Data untuk aplikasi           laporan final 116 tanggal test
```

Alur ini mengikuti masukan mentor (lapisan rule-based classification dan deterministic validation) dan advisor (observability).

## Cell 1: Muat model dan buktikan hasilnya sama dengan Notebook 2

**Tujuan:** memastikan model yang dipakai untuk laporan GenAI adalah model yang sama dengan yang dievaluasi di Notebook 2.

Fungsi `buat_fitur_model`, `beri_label`, dan `policy` ditulis ulang **sama persis** dengan Notebook 2, lalu model memprediksi periode test.

**Hasil pengecekan:**
- versi scikit-learn sama (1.6.1),
- nama dan urutan fitur sama,
- F1 macro sekarang **0,828**, identik dengan yang tersimpan.

Jika salah satu pengecekan ini gagal, laporan GenAI tidak bisa dipercaya karena berasal dari model yang berbeda.

In [ ]:
#Cell 1 : setup , muat model dan data, buktikan hasilnya sama dengan notebook model

from google.colab import drive
from pathlib import Path
import json
import joblib
import numpy as np
import pandas as pd
import sklearn
from sklearn.metrics import f1_score

drive.mount("/content/drive")

#1 folder project
PROJECT_DIR = Path("/content/drive/MyDrive/Tewiwi-Bakehouse")
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"
MODEL_DIR = PROJECT_DIR / "models"
OUTPUT_DIR = PROJECT_DIR / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

#2. Muat model beserta "kartu identitasnya"
bundle = joblib.load(MODEL_DIR / "usage_model_gb_q07.joblib")
model = bundle["model"]
FITUR_MODEL= bundle["fitur_model"]
CFG = bundle["config"]
H = CFG["H"]
SEED = CFG["SEED"]
RESTOCK_DAYS = CFG["RESTOCK_DAYS"]
EXCESS_DAYS = CFG["EXCESS_DAYS"]
LABELS = ["Restock","Sufficient","Excess"]

print("Versi sckit-learn saat model disimpan :", bundle["sklearn_version"])
print("Versi sckit-learn sekarang", sklearn.__version__)

#3. Muat dataset
data = pd.read_csv(PROCESSED_DIR / "dataset_model.csv", parse_dates=["date"])

#4. Tulis ulang fungsi, Harus sama persis dengan notebook modeling
def buat_fitur_model(df):
  X = pd.DataFrame(index=df.index)
  X["tren_7_vs_30"] = df["avg_usage_7"] / df["avg_usage_30"]
  X["stok_cover_now"] = df["stok_cover_now"]
  X["on_order_hari"] = df["on_order"] / df["avg_usage_30"]
  X["posisi_stok"] = df["posisi_stok"]
  X["days_since_last_order"] = df["days_since_last_order"]
  X["bulan"] = df["date"].dt.month
  return X

def beri_label(sisa_days):
  return np.select(
      [sisa_days < RESTOCK_DAYS, sisa_days > EXCESS_DAYS],
      ["Restock","Excess"],
      default="Sufficient"
  )
def policy(df, pred_usage):
  proyeksi = df["stok_now"] + df["on_order"] - pred_usage
  sisa_days = proyeksi /df["avg_usage_30"]
  return sisa_days, beri_label(sisa_days)

#5. Pastikan nama dan urutan fitur sama dengan yang dipaki saat melatih
fitur_cek = list(buat_fitur_model(data.head()).columns)
print("\nFitur sama dengan saat training :", fitur_cek == FITUR_MODEL)

#6. prediksi untuk periode test
test = data[data["split"]== "test"].copy()
test["pred_rasio"] = model.predict(buat_fitur_model(test)[FITUR_MODEL])
test["pred_usage"] = test["pred_rasio"] * test["avg_usage_30"]
test["pred_sisa_days"], test["pred_label"] = policy(test, test["pred_usage"])

#7. BUktikan : hasil harus sama dengan notebook modeling
f1_sekarang = f1_score(test["label"], test["pred_label"], labels=LABELS, average="macro")
f1_tersimpan = bundle["metrik_test"]["GB quantile 0.7"]["f1_macro"]
print(f"F1 Macro sekarang: {f1_sekarang:.3f}")
print(f"F1 Macro saat disimpan: {f1_tersimpan:.3f}")
print("Hasil identik:", round(f1_sekarang,3) == round(f1_tersimpan,3 ))

Mounted at /content/drive
Versi sckit-learn saat model disimpan : 1.6.1
Versi sckit-learn sekarang 1.6.1

Fitur sama dengan saat training : True
F1 Macro sekarang: 0.828
F1 Macro saat disimpan: 0.828
Hasil identik: True


## Cell 2: Inventory snapshot, satu-satunya input untuk GenAI

**Tujuan:** menyusun tabel kondisi 10 bahan pada satu tanggal, hanya dari informasi yang tersedia **pagi hari** ditambah prediksi model.

| Kolom | Arti |
|---|---|
| `stok_gudang` | stok di gudang pagi ini |
| `sedang_dikirim` | bahan yang sudah dipesan tetapi belum datang |
| `perkiraan_pakai_7_hari` | perkiraan pemakaian 7 hari dari model |
| `sisa_setelah_7_hari` | stok tersisa di akhir 7 hari bila tidak memesan lagi (negatif berarti kurang) |
| `kurang_jika_tidak_pesan` | besar kekurangan bila tidak memesan lagi |
| `sisa_dalam_hari` | sisa di akhir 7 hari, setara berapa hari pemakaian rata-rata |
| `status` | label dari policy: Perlu restok, Cukup, Berlebih |

**Keputusan penting:**
- Semua perhitungan dilakukan di sini, sehingga LLM **tidak perlu berhitung**.
- Satuan dibuat mudah dibaca owner: telur dalam **butir** (dibulatkan ke bilangan bulat), bahan lain dalam **kg** (1 desimal).

**Pengecekan:** status konsisten dengan ambang hari, dan semua angka telur sudah bilangan bulat.

In [ ]:
# Cell 2 : Buat snapshot kondisi stok per tanggal (satu-satunya input untuk GenAi)
STATUS_ID = {
    "Restock" : "Perlu restok",
    "Sufficient" : "Cukup",
    "Excess" : "Berlebih"
}
URUTAN_STATUS={STATUS_ID["Restock"]:0, STATUS_ID["Excess"]:1, STATUS_ID["Sufficient"]:2}

def buat_snapshot(df, tanggal):
  """ table kondisi 10 bahan pada satu tanggal , hanya dari info pagi hari + prediksi model """
  d = df[df["date"] == pd.Timestamp(tanggal)].copy()

  #1. telur dalam butir , bahan lain diubah dari gram ke kg
  faktor = np.where(d["bahan"] == "telur", 1.0, 0.001)

  #2. hitungan yang sudah di siapkan , supaya gemini tidak berhitung
  tersedia = d["stok_now"] + d["on_order"]
  sisa = tersedia - d["pred_usage"]
  kurang = (-sisa).clip(lower=0)

  def bulat(nilai):
    return np.where(d["bahan"] == "telur", np.round(nilai,0), np.round(nilai,1))

  snap = pd.DataFrame({
      "bahan"                   :d["bahan"].str.replace("_"," ").str.title().values,
      "satuan"                  :np.where(d["bahan"] == "telur", "butir", "kg"),
      "stok_gudang"             :bulat(d["stok_now"] * faktor),
      "sedang_dikirim"          :bulat(d["on_order"]* faktor),
      "perkiraan_pakai_7_hari"  :bulat(d["pred_usage"]*faktor),
      "sisa_setelah_7_hari"     :bulat(sisa*faktor),
      "kurang_jika_tidak_pesan" :bulat(kurang * faktor),
      "sisa_dalam_hari"         :d["pred_sisa_days"].round(1).values,
      "status"                  :d["pred_label"].map(STATUS_ID).values,
  })
  # 3. Urutkan: yang paling butuh perhatian di atas
  snap["_urut"] = snap["status"].map(URUTAN_STATUS)
  snap = snap.sort_values(["_urut", "sisa_dalam_hari"]).drop(columns="_urut")
  return snap.reset_index(drop=True)

# contoh: hari pertama periode test
contoh_tanggal = test["date"].min()
snap_contoh =buat_snapshot(test, contoh_tanggal)

print("Snapshot per", contoh_tanggal.date())
display(snap_contoh)
print("Jumlah per status:", snap_contoh["status"].value_counts().to_dict())

# Cek konsistensi : status harus sesuai ambang sisa dalam hari
cek_restock = (snap_contoh.loc[snap_contoh["status"] == STATUS_ID["Restock"], "sisa_dalam_hari"] < RESTOCK_DAYS).all()
cek_excess = (snap_contoh.loc[snap_contoh["status"]==STATUS_ID["Excess"], "sisa_dalam_hari"] > EXCESS_DAYS).all()
print("Status konsisten dengan ambang :", cek_restock and cek_excess)

# Cek pembulatan telur : semua angka telur harus bilangan bulat
baris_telur = snap_contoh[snap_contoh["satuan"] == "butir"]
kolom_harga = ["stok_gudang","sedang_dikirim","perkiraan_pakai_7_hari","sisa_setelah_7_hari","kurang_jika_tidak_pesan"]
print("Telur sudah butir utuh:", (baris_telur[kolom_harga] % 1 == 0 ).all().all())

Snapshot per 2022-06-01


,bahan,satuan,stok_gudang,sedang_dikirim,perkiraan_pakai_7_hari,sisa_setelah_7_hari,kurang_jika_tidak_pesan,sisa_dalam_hari,status
0,Tepung Gandum Utuh,kg,75.3,0.0,95.5,-20.2,20.2,-1.6,Perlu restok
1,Mentega,kg,16.8,0.0,20.2,-3.4,3.4,-1.3,Perlu restok
2,Gula,kg,2.4,18.4,20.2,0.6,0.0,0.2,Cukup
3,Telur,butir,17.0,130.0,141.0,6.0,0.0,0.4,Cukup
4,Garam,kg,3.0,12.1,11.9,3.3,0.0,2.0,Cukup
5,Terigu Protein Tinggi,kg,119.6,495.2,483.3,131.5,0.0,2.0,Cukup
6,Ragi Instan,kg,0.9,1.5,1.4,1.0,0.0,5.8,Cukup
7,Cream Cheese,kg,7.0,11.3,10.6,7.8,0.0,5.9,Cukup
8,Susu Bubuk,kg,3.0,4.6,4.4,3.2,0.0,6.0,Cukup
9,Cokelat,kg,8.3,13.0,12.2,9.1,0.0,6.1,Cukup


Jumlah per status: {'Cukup': 8, 'Perlu restok': 2}
Status konsisten dengan ambang : True
Telur sudah butir utuh: True


## Cell 2b: Klasifikasi rule-based (saran mentor)

**Tujuan:** menentukan untuk setiap bahan, **sebelum** LLM dipanggil:
- masuk bagian laporan mana (Perlu dibeli, Berpotensi berlebih, Perlu dipantau, atau tidak disebut),
- jenis nilai yang ditampilkan (kurang atau sisa),
- angka dan satuannya,
- baris laporan yang sudah jadi, misalnya `- Gula: kurang 9,3 kg`.

| Status | Aturan | Bagian laporan |
|---|---|---|
| Perlu restok | selalu | Perlu dibeli, dengan nilai **kurang** |
| Berlebih | selalu | Berpotensi berlebih, dengan nilai **sisa** |
| Cukup | `sisa_dalam_hari` < 1 hari (`PANTAU_DAYS`) | Perlu dipantau, dengan nilai **sisa** |
| Cukup | lainnya | tidak disebut (aman) |

**Kenapa lapisan ini dibuat?** Pada uji awal, LLM diberi tabel lengkap dan diminta memilah sendiri. Hasilnya, LLM beberapa kali **memindahkan bahan berstatus Cukup ke bagian Berpotensi berlebih** dan **melewatkan bahan yang perlu dipantau**. Memilah bahan adalah keputusan logika, sehingga lebih tepat dikerjakan oleh kode.

In [ ]:
# CELL 2b : Klasifikasi rule-based (Python): tentukan bagian, jenis nilai, angka, dan satuan

PANTAU_DAYS = 1   # status Cukup tapi sisa_dalam_hari di bawah ini → "Perlu dipantau"

URUTAN_BAGIAN = ["Perlu dibeli", "Berpotensi berlebih", "Perlu dipantau"]

def format_angka(nilai, satuan):
    """Angka gaya Indonesia: telur bilangan bulat, bahan lain 1 desimal dengan koma."""
    if satuan == "butir":
        return f"{nilai:.0f}"
    return f"{nilai:.1f}".replace(".", ",")

def klasifikasi_rule(snap):
    """Semua keputusan (bagian, angka, satuan) dibuat oleh KODE. LLM nanti hanya menyalin dan menjelaskan."""
    k = snap.copy()

    # 1. Bagian laporan
    k["bagian"] = np.select(
        [k["status"] == STATUS_ID["Restock"],
         k["status"] == STATUS_ID["Excess"],
         (k["status"] == STATUS_ID["Sufficient"]) & (k["sisa_dalam_hari"] < PANTAU_DAYS)],
        URUTAN_BAGIAN,
        default="-",
    )

    # 2. Jenis nilai dan angka yang boleh ditampilkan untuk tiap bahan
    k["jenis_nilai"] = np.select(
        [k["bagian"] == "Perlu dibeli", k["bagian"].isin(["Berpotensi berlebih", "Perlu dipantau"])],
        ["kurang", "sisa"],
        default="-",
    )
    k["nilai"] = np.where(k["jenis_nilai"] == "kurang", k["kurang_jika_tidak_pesan"],
                 np.where(k["jenis_nilai"] == "sisa", k["sisa_setelah_7_hari"], np.nan))

    # 3. Baris laporan yang sudah jadi (angka + satuan sudah benar)
    def susun_baris(r):
        if r["jenis_nilai"] == "-":
            return ""
        teks = f"- {r['bahan']}: {r['jenis_nilai']} {format_angka(r['nilai'], r['satuan'])} {r['satuan']}"
        return teks + (" setelah 7 hari" if r["jenis_nilai"] == "sisa" else "")
    k["baris_laporan"] = k.apply(susun_baris, axis=1)
    return k

# Contoh: tanggal pertama periode test
klas_contoh = klasifikasi_rule(snap_contoh)
display(klas_contoh[["bahan", "status", "sisa_dalam_hari", "bagian", "jenis_nilai", "nilai", "satuan", "baris_laporan"]])

# Cek: setiap bahan Perlu restok wajib masuk "Perlu dibeli", setiap Berlebih wajib masuk "Berpotensi berlebih"
cek_1 = (klas_contoh.loc[klas_contoh["status"] == STATUS_ID["Restock"], "bagian"] == "Perlu dibeli").all()
cek_2 = (klas_contoh.loc[klas_contoh["status"] == STATUS_ID["Excess"], "bagian"] == "Berpotensi berlebih").all()
cek_3 = (klas_contoh.loc[klas_contoh["bagian"] != "-", "baris_laporan"] != "").all()
print("Aturan klasifikasi konsisten:", cek_1 and cek_2 and cek_3)

,bahan,status,sisa_dalam_hari,bagian,jenis_nilai,nilai,satuan,baris_laporan
0,Tepung Gandum Utuh,Perlu restok,-1.6,Perlu dibeli,kurang,20.2,kg,"- Tepung Gandum Utuh: kurang 20,2 kg"
1,Mentega,Perlu restok,-1.3,Perlu dibeli,kurang,3.4,kg,"- Mentega: kurang 3,4 kg"
2,Gula,Cukup,0.2,Perlu dipantau,sisa,0.6,kg,"- Gula: sisa 0,6 kg setelah 7 hari"
3,Telur,Cukup,0.4,Perlu dipantau,sisa,6.0,butir,- Telur: sisa 6 butir setelah 7 hari
4,Garam,Cukup,2.0,-,-,NaN,kg,
5,Terigu Protein Tinggi,Cukup,2.0,-,-,NaN,kg,
6,Ragi Instan,Cukup,5.8,-,-,NaN,kg,
7,Cream Cheese,Cukup,5.9,-,-,NaN,kg,
8,Susu Bubuk,Cukup,6.0,-,-,NaN,kg,
9,Cokelat,Cukup,6.1,-,-,NaN,kg,


Aturan klasifikasi konsisten: True


## Cell 3: Tiga tanggal contoh untuk menyetel prompt

**Tujuan:** memilih tiga kondisi yang berbeda untuk melihat perilaku LLM:

| Kondisi | Tanggal | Isi |
|---|---|---|
| Kritis | 10 Juli 2022 | 10 bahan Perlu restok |
| Berlebih | 10 September 2022 | 5 bahan Berlebih, 5 Cukup |
| Aman | 18 Juni 2022 | 10 bahan Cukup |

Ketepatan prediksi model di tanggal ini dicetak **hanya untuk kita** dan tidak diberikan ke LLM.

**Penting untuk evaluasi:** karena prompt disetel dengan melihat tiga tanggal ini, ketiganya **dikeluarkan** dari evaluasi di Cell 7 supaya penilaian tetap adil.

In [ ]:
# Cell 3 : Pilih tanggal contoh untuk menguji ringkasan genAI

# 1. HItung jumlah prediksi tiap status per tanggal di periode test
per_tanggal = (
    test.groupby("date")["pred_label"]
    .value_counts()
    .unstack(fill_value=0)
    .reindex(columns=LABELS, fill_value=0)
)

print("==== Sebaran jumlah bahan per status dalam satu hari (periode set)====")

# 2. Pilih tiga tanggal dengan kondisi berbeda
TGL_KRITIS = per_tanggal["Restock"].idxmax()
TGL_BERLEBIH = per_tanggal["Excess"].idxmax()
HARI_AMAN = per_tanggal[(per_tanggal["Restock"] == 0 ) & (per_tanggal["Excess"] == 0)]
TGL_AMAN = HARI_AMAN.index[0] if len(HARI_AMAN) > 0 else per_tanggal["Restock"].idxmin()

TANGGAL_CONTOH = {"kritis": TGL_KRITIS, "berlebih": TGL_BERLEBIH, "aman": TGL_AMAN}
print("\nJumlah hari tanpa Restock dan tanpa Excess:", len(HARI_AMAN))


#3. Tampilkan snapshot tiap tanggal + ketepatan prediksi model (Tidak diberikan ke AI)
for nama, tgl in TANGGAL_CONTOH.items():
  snap = buat_snapshot(test, tgl)
  hari =  test[test["date"] == tgl]
  tepat =  (hari["pred_label"] == hari["label"]).mean() *100

  print(f"\n==== KOndisi {nama.upper()}: {tgl.date()}====")
  print("Jumlah per status:", snap["status"].value_counts().to_dict())
  print(f"Prediksi model tepat untuk {tepat:.0f}% bahan di tanggal ini (info untuk kita saja)")
  display(snap)

==== Sebaran jumlah bahan per status dalam satu hari (periode set)====

Jumlah hari tanpa Restock dan tanpa Excess: 14

==== KOndisi KRITIS: 2022-07-10====
Jumlah per status: {'Perlu restok': 10}
Prediksi model tepat untuk 100% bahan di tanggal ini (info untuk kita saja)


,bahan,satuan,stok_gudang,sedang_dikirim,perkiraan_pakai_7_hari,sisa_setelah_7_hari,kurang_jika_tidak_pesan,sisa_dalam_hari,status
0,Susu Bubuk,kg,2.7,0.0,6.5,-3.8,3.8,-6.6,Perlu restok
1,Mentega,kg,13.7,0.0,26.6,-12.9,12.9,-4.9,Perlu restok
2,Ragi Instan,kg,1.0,0.0,1.9,-0.9,0.9,-4.8,Perlu restok
3,Cokelat,kg,9.1,0.0,16.2,-7.2,7.2,-4.5,Perlu restok
4,Garam,kg,10.0,0.0,16.4,-6.4,6.4,-3.8,Perlu restok
5,Terigu Protein Tinggi,kg,412.7,0.0,669.9,-257.2,257.2,-3.8,Perlu restok
6,Gula,kg,20.7,0.0,30.0,-9.3,9.3,-3.7,Perlu restok
7,Telur,butir,139.0,0.0,197.0,-58.0,58.0,-3.3,Perlu restok
8,Cream Cheese,kg,9.7,0.0,13.5,-3.8,3.8,-2.8,Perlu restok
9,Tepung Gandum Utuh,kg,40.8,82.0,128.9,-6.1,6.1,-0.5,Perlu restok



==== KOndisi BERLEBIH: 2022-09-10====
Jumlah per status: {'Berlebih': 5, 'Cukup': 5}
Prediksi model tepat untuk 100% bahan di tanggal ini (info untuk kita saja)


,bahan,satuan,stok_gudang,sedang_dikirim,perkiraan_pakai_7_hari,sisa_setelah_7_hari,kurang_jika_tidak_pesan,sisa_dalam_hari,status
0,Cokelat,kg,31.7,0.0,11.7,20.0,0.0,7.1,Berlebih
1,Cream Cheese,kg,30.1,0.0,10.8,19.3,0.0,7.4,Berlebih
2,Ragi Instan,kg,1.5,2.3,1.2,2.6,0.0,8.1,Berlebih
3,Garam,kg,36.2,0.0,12.1,24.2,0.0,8.2,Berlebih
4,Terigu Protein Tinggi,kg,512.8,935.1,458.6,989.3,0.0,8.2,Berlebih
5,Susu Bubuk,kg,7.2,0.0,3.8,3.4,0.0,3.5,Cukup
6,Mentega,kg,33.6,0.0,16.3,17.2,0.0,4.1,Cukup
7,Tepung Gandum Utuh,kg,197.8,0.0,84.7,113.2,0.0,5.1,Cukup
8,Gula,kg,41.6,0.0,16.8,24.9,0.0,5.8,Cukup
9,Telur,butir,304.0,0.0,118.0,186.0,0.0,6.2,Cukup



==== KOndisi AMAN: 2022-06-18====
Jumlah per status: {'Cukup': 10}
Prediksi model tepat untuk 80% bahan di tanggal ini (info untuk kita saja)


,bahan,satuan,stok_gudang,sedang_dikirim,perkiraan_pakai_7_hari,sisa_setelah_7_hari,kurang_jika_tidak_pesan,sisa_dalam_hari,status
0,Telur,butir,119.0,0.0,116.0,3.0,0.0,0.1,Cukup
1,Gula,kg,16.9,0.0,16.3,0.5,0.0,0.2,Cukup
2,Terigu Protein Tinggi,kg,532.0,0.0,432.8,99.2,0.0,1.4,Cukup
3,Garam,kg,13.1,0.0,10.6,2.6,0.0,1.5,Cukup
4,Cokelat,kg,18.0,0.0,12.4,5.7,0.0,3.2,Cukup
5,Ragi Instan,kg,2.2,0.0,1.4,0.8,0.0,4.0,Cukup
6,Mentega,kg,11.2,21.9,17.3,15.9,0.0,5.3,Cukup
7,Cream Cheese,kg,18.5,0.0,9.5,9.0,0.0,5.9,Cukup
8,Tepung Gandum Utuh,kg,59.4,91.3,73.0,77.7,0.0,6.0,Cukup
9,Susu Bubuk,kg,2.6,4.8,3.6,3.8,0.0,6.4,Cukup


## Cell 4: Koneksi ke penyedia LLM dan Langfuse

**Tujuan:** menghubungkan notebook ke LLM dan ke Langfuse (pencatat setiap panggilan LLM untuk observability).

**Keamanan:** semua API key disimpan di **Colab Secrets** dan dibaca dengan `userdata.get(...)`. Tidak ada key yang ditulis di kode, markdown, atau output.

**Format OpenAI-compatible:** semua penyedia di `PENYEDIA` memakai format yang sama, sehingga berganti penyedia cukup dengan mengubah `PILIHAN`.

### Perjalanan pemilihan model (keputusan berbasis data)

| Penyedia dan model | Kendala |
|---|---|
| Z.ai, GLM-4.7-Flash (gratis) | server sering penuh (kode 1305). Log mencatat **error rate 83,9%** dan median **105,6 detik** per ringkasan |
| OpenRouter, GLM-5.3-Flash `:free` | versi gratis dihentikan oleh penyedia |
| OpenRouter dan Google AI Studio, Gemma 4 31B | kuota bersama habis (429) dan server sibuk (503) |
| **Groq, gpt-oss-120b** | **dipakai**: latency median sekitar 2 detik, error rate di bawah 1% |

Pergantian model dilakukan karena **angka observability**, bukan karena perasaan. Pengalaman ini juga menunjukkan risiko layanan gratis: model bisa dihentikan atau penuh tanpa pemberitahuan, sehingga sistem membutuhkan guardrail (Cell 6b).

**Catatan output:** peringatan merah dari `pip` tentang `google-adk` dan `opentelemetry` berasal dari library bawaan Colab yang **tidak dipakai** di notebook ini, sehingga tidak memengaruhi hasil.

In [ ]:
# CELL 4 : Hubungkan ke Apikey AI
import os
from openai import OpenAI
from google.colab import userdata

!pip install -q langfuse "opentelemetry-api==1.45.2" "opentelemetry-sdk==1.45.2"

# 1. Langfuse: mencatat setiap panggilan LLM (prompt, jawaban, latency, token, error)
LANGFUSE_AKTIF = True
LANGFUSE_HOST  = "https://hipaa.cloud.langfuse.com"     # region akun Langfuse-mu

if LANGFUSE_AKTIF:
  os.environ["LANGFUSE_PUBLIC_KEY"] = userdata.get("langfuse_public")
  os.environ["LANGFUSE_SECRET_KEY"] = userdata.get("langfuse_secret")
  os.environ["LANGFUSE_HOST"]       = LANGFUSE_HOST
  os.environ["LANGFUSE_BASE_URL"]   = LANGFUSE_HOST   # nama variabel di versi library yang lebih baru
  from langfuse.openai import OpenAI                  # client OpenAI yang sudah "dibungkus" Langfuse
  from langfuse import get_client
  print("Langfuse terhubung:", get_client().auth_check())
else:
    from openai import OpenAI

# 2. Daftar penyedia yang bisa dipakai (semua memakai format OpenAI)
PENYEDIA = {
    "glm":  {"base_url": "https://api.z.ai/api/paas/v4/",  "secret": "glm",  "model": "glm-4.7-flash"},
    "groq": {"base_url": "https://api.groq.com/openai/v1", "secret": "groq", "model": "openai/gpt-oss-120b"},
    "openrouter": {"base_url": "https://openrouter.ai/api/v1",   "secret": "openrouter", "model": "google/gemma-4-31b-it:free"},
    "gemma": {"base_url": "https://generativelanguage.googleapis.com/v1beta/openai/", "secret": "gem", "model": "models/gemma-4-31b-it"},
}

# 3. Pilih AI
PILIHAN = "groq"

# 4. Pengaturan jawaban : temperatur rendah supaya jawaban konsisten dan tidak"Kreatif"
cfg = PENYEDIA[PILIHAN]
client = OpenAI(
    api_key=userdata.get(cfg["secret"]),
    base_url=cfg["base_url"],
    timeout=120,       # maksimal tunggu 2 menit per jawaban
    max_retries=0,     # percobaan ulang diatur sendiri di fungsi ringkas()
)
MODEL_GENAI = cfg["model"]
TEMPERATURE = 0.2

# 5. Uji koneksi (memakai 1 kuota). Nyalakan True hanya saat perlu mengecek
UJI_KONEKSI = False
if UJI_KONEKSI:
    resp = client.chat.completions.create(
        model=MODEL_GENAI,
        messages=[{"role": "user", "content": "Jawab dengan satu kata saja: berapa 2 ditambah 3?"}],
        temperature=TEMPERATURE,
    )
    print("Jawaban:", resp.choices[0].message.content)

print("Penyedia:", PILIHAN, "| Model:", MODEL_GENAI, "| Langfuse:", LANGFUSE_AKTIF)
# print(sorted(m.id for m in client.models.list()))

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.45.2 which is incompatible.
google-adk 2.7.1 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.45.2 which is incompatible.
Langfuse terhubung: True
Penyedia: groq | Model: openai/gpt-oss-120b | Langfuse: True


## Cell 4b: Cache jawaban LLM

**Tujuan:** menyimpan setiap jawaban LLM ke `cache_ringkasan.json`.

**Kunci cache** = nama model +hash MD5 isi prompt. Akibatnya:
- menjalankan ulang notebook **tidak memanggil LLM lagi**, sehingga tidak memakai kuota dan hasilnya bisa direproduksi;
- jika prompt atau model berubah, kuncinya berubah, sehingga jawaban lama tidak tertukar dengan jawaban baru.

In [ ]:
# CELL 4b : Cache jawaban LLM, supaya menjalankan ulang notebook tidak memakai kuota

import json
import hashlib
import time

CACHE_FILE = OUTPUT_DIR / "cache_ringkasan.json"

# Baca cache lama kalau ada, kalau belum ada mulai dari kosong
CACHE = json.loads(CACHE_FILE.read_text()) if CACHE_FILE.exists() else {}
print("Jumlah jawaban tersimpan di cache:", len(CACHE))

def simpan_cache():
    CACHE_FILE.write_text(json.dumps(CACHE, ensure_ascii=False, indent=1))

class KuotaHabis(Exception):
    """Tanda bahwa kuota atau batas permintaan habis, supaya proses bisa berhenti dengan rapi."""
    pass

Jumlah jawaban tersimpan di cache: 86


## Cell 5: GenAI Explanation (prompt v3) dan pencatatan observability

### Isi prompt v3
LLM hanya menerima: tanggal, jumlah bahan per status, dan **daftar baris yang sudah final** dari Cell 2b. Tugasnya:
1. menulis satu kalimat pembuka berisi jumlah bahan per status,
2. **menyalin persis** ketiga bagian laporan,
3. menulis satu kalimat saran dengan kata-katanya sendiri: memesan semua bahan Perlu dibeli, menunda bahan Berpotensi berlebih, memantau (bukan membeli) bahan Perlu dipantau.

LLM dilarang menyebut harga, pemasok, cuaca, acara, atau alasan apa pun, karena informasi itu tidak ada di data.

### Evolusi prompt
| Versi | Isi yang diberikan ke LLM | Masalah |
|---|---|---|
| v1 | tabel lengkap, LLM memilah sendiri | bahan salah masuk bagian, bahan pantau terlewat |
| v2 | tabel + kolom `masuk_bagian` dari kode | masih harus memilih angka dan satuan sendiri |
| **v3** | **hanya baris laporan yang sudah jadi** | dipakai dan **dibekukan** untuk evaluasi |

### Observability (saran advisor)
Setiap panggilan LLM dicatat ke `log_llm.csv`: waktu, model, versi prompt, latency, total waktu termasuk percobaan ulang, jenis error, dan jumlah token. Panggilan yang sama juga dikirim ke **Langfuse**, sehingga setiap prompt dan jawaban bisa dibuka di dashboard.

### Penanganan gangguan
- Server penuh, batas kecepatan, atau waktu habis: tunggu makin lama (30, 60, 90 detik, dan seterusnya), lalu coba lagi.
- Kuota harian habis: berhenti langsung, karena menunggu tidak membantu.
- Parameter `ulang` dipakai Cell 6b untuk meminta **jawaban baru** jika jawaban pertama gagal validasi.

In [ ]:
# CELL 5 : Rancang prompt dan buat ringkasan untuk tiga tanggal contoh
PROMPT_VERSI = "v3"
LOG_FILE = OUTPUT_DIR / "log_llm.csv"
PANTAU_DAYS = 1   # status Cukup tapi sisa_dalam_hari di bawah ini → "Perlu dipantau"

BULAN_ID = ["Januari", "Februari", "Maret", "April", "Mei", "Juni",
            "Juli", "Agustus", "September", "Oktober", "November", "Desember"]

def format_tanggal(tgl):
    tgl = pd.Timestamp(tgl)
    return f"{tgl.day} {BULAN_ID[tgl.month - 1]} {tgl.year}"

# ---------  1. Susun daftar per bagiaan dari hasil klasifikasi (dipakai juga oleh laporan template) --------
def susun_daftar(klas):
    blok = []
    for bagian in URUTAN_BAGIAN:
        baris = klas.loc[klas["bagian"] == bagian, "baris_laporan"].tolist()
        isi = "\n".join(baris) if baris else "Tidak ada"
        blok.append(f"**{bagian}:**\n{isi}")
    return "\n\n".join(blok)

def jumlah_per_status(snap):
    jumlah = snap["status"].value_counts()
    return ", ".join(f"{status} = {n} bahan" for status, n in jumlah.items())

# --------- 2. Prompt v3: pendek , LLM tidak memilah dan tidak memilih angka ----------
def buat_prompt(snap, tanggal):
  klas = klasifikasi_rule(snap)
  return f"""Kamu adalah asisten yang menulis laporan stok bahan baku untuk pemilik Tewiwi Bakehouse

TANGGAL:{format_tanggal(tanggal)}
JUMLAH BAHAN PER STATUS:{jumlah_per_status(snap)}
DAFTAR BAHAN PER BAGIAN (sudah final, ditentukan oleh sistem)
{susun_daftar(klas)}

ARTI ISTILAH :
- kurang: jumlah kekurangan bila tidak memesan lagi dalam 7 hari ke depan.
- sisa ... setelah 7 hari: sisa stok di akhir 7 hari bila tidak memesan lagi.
- Berpotensi berlebih: setelah 7 hari, stok masih tersisa lebih dari 7 hari pemakaian
- Perlu dipantau: stok masih cukup, tetapi sisanya tipis di akhir 7hari.

TULIS LAPORAN DENGAN SUSUNAN BERIKUT:
1. Satu kalimat pembuka yang menyebut jumlah bahan per status persis seperti JUMLAH BAHAN PER STATUS, tanpa menulis label "JUMLAH BAHAN PER STATUS".
2. Ketiga bagian disalin PERSIS seperti DAFTAR BAHAN PER BAGIAN: judul, setiap baris, dan tulisan "Tidak ada".
   Jangan mengubah nama bahan, angka, satuan, atau urutan. Jangan menambah, menghapus, atau memindahkan baris.
3. Satu kalimat penutup berisi saran langsung kepada pemilik:
   - Bahan di bagian "Perlu dibeli": sarankan memesan SEMUA bahan itu.
   - Bahan di bagian "Berpotensi berlebih": sarankan menunda pembelian bahan itu.
   - Bahan di bagian "Perlu dipantau": sarankan MEMANTAU, bukan membeli.
   - Jika ketiga bagian "Tidak ada": katakan stok aman untuk 7 hari ke depan.

ATURAN:
- Kalimat penutup ditulis dengan kata-katamu sendiri, tidak menyalin teks aturan, dan tidak menyebut angka apapun kecuali "7 hari".
- Dikalimat penutup, hanya sebut nama bahan yang ada di DAFTAR BAHAN PER BAGIAN
- Dilarang menyebut harga, pemasok, merek, cuaca, acara, atau alasan penyebab apa pun.
- Gunakan bahasa Indonesia yang sederhana, seperti berbicara kepada pemilik usaha kecil.
- Pisahkan kalimat pembuka, setiap bagian, dan kalimat penutup dengan satu baris kosong.
- Maksimal 200 kata.

"""

# ------ 3. Log setiap panggilan LLM ---------
def jenis_error(pesan):
    if "per-day" in pesan or "per day" in pesan:
        return "kuota_harian"            # batas harian OpenRouter free: menunggu tidak membantu
    if "1305" in pesan or "overloaded" in pesan:
        return "server_penuh"
    if "429" in pesan or "rate limit" in pesan or "rate_limit" in pesan or "resource_exhausted" in pesan:
        return "batas_kecepatan"
    if "timed out" in pesan or "timeout" in pesan:
        return "waktu_habis"
    if any(k in pesan for k in ["500", "502", "503", "unavailable"]):
        return "server_error"
    return "lainnya"

KOLOM_LOG = ["waktu", "tanggal_data", "model", "versi_prompt", "ulang", "status", "latency_detik",
             "total_detik", "n_error", "jenis_error", "token_input", "token_output"]

def catat_log(data):
    pd.DataFrame([data]).reindex(columns=KOLOM_LOG).to_csv(
        LOG_FILE, mode="a", header=not LOG_FILE.exists(), index=False)

# ----------------------- 4. Panggil LLM (cache, percobaan ulang, log) -----------------------
def ringkas(snap, tanggal, coba=4, ulang=0):
    """ulang = 0 untuk jawaban pertama. ulang = 1, 2, ... untuk meminta jawaban BARU
    (dipakai Cell 6b saat jawaban pertama gagal validasi), supaya tidak diambil dari cache yang sama."""
    prompt = buat_prompt(snap, tanggal)

    kunci = MODEL_GENAI + "|" + hashlib.md5(prompt.encode("utf-8")).hexdigest()
    if ulang > 0:
        kunci += f"|ulang{ulang}"
    if kunci in CACHE:
        return CACHE[kunci]

    log = {
        "waktu":        pd.Timestamp.now(tz="Asia/Jakarta").strftime("%Y-%m-%d %H:%M:%S"),
        "tanggal_data": str(pd.Timestamp(tanggal).date()),
        "model":        MODEL_GENAI,
        "versi_prompt": PROMPT_VERSI,
        "ulang":        ulang,
    }

    # Label trace untuk Langfuse (hanya dikirim lewat OpenRouter; penyedia lain bisa menolak field ini)
    ekstra = {}
    if PILIHAN == "openrouter":
        ekstra = {
            "user": "tewiwi-capstone",
            "extra_body": {
                "session_id": f"evaluasi-{PROMPT_VERSI}",
                "trace": {
                    "trace_name":      "laporan_stok_tewiwi",
                    "generation_name": f"ringkasan_{PROMPT_VERSI}_ulang{ulang}",
                    "environment":     "evaluasi",
                    "release":         PROMPT_VERSI,
                    "tanggal_data":    log["tanggal_data"],
                },
            },
        }
    # Label trace untuk Langfuse (dipakai oleh client yang dibungkus Langfuse di Cell 4)
    if LANGFUSE_AKTIF:
        ekstra.update({
            "name": f"ringkasan_{PROMPT_VERSI}_ulang{ulang}",
            "metadata": {
                "langfuse_session_id": f"evaluasi-{PROMPT_VERSI}",
                "langfuse_user_id":    "tewiwi-capstone",
                "langfuse_tags":       [PROMPT_VERSI, MODEL_GENAI],
                "tanggal_data":        log["tanggal_data"],
                "ulang":               ulang,
            },
        })
    mulai_total = time.time()
    daftar_error = []

    for i in range(coba):
        mulai = time.time()
        try:
            resp = client.chat.completions.create(
                model=MODEL_GENAI,
                messages=[{"role": "user", "content": prompt}],
                temperature=TEMPERATURE,
                **ekstra,
            )
            latency = time.time() - mulai
            break
        except Exception as e:
            jenis = jenis_error(str(e).lower())
            daftar_error.append(jenis)

            # Kuota harian habis: berhenti langsung, jangan buang waktu menunggu
            boleh_ulang = jenis not in ["lainnya", "kuota_harian"]
            if boleh_ulang and i < coba - 1:
                tunggu = 30 * (i + 1)
                print(f"  {jenis}, coba lagi dalam {tunggu} detik..")
                time.sleep(tunggu)
                continue

            catat_log({**log, "status": "gagal",
                       "total_detik": round(time.time() - mulai_total, 1),
                       "n_error": len(daftar_error), "jenis_error": ";".join(daftar_error)})
            if jenis in ["kuota_harian", "server_penuh", "batas_kecepatan"]:
                raise KuotaHabis(f"LLM tidak tersedia: {jenis}")
            raise

    teks  = resp.choices[0].message.content or ""
    usage = getattr(resp, "usage", None)
    catat_log({**log, "status": "sukses" if teks else "kosong",
               "latency_detik": round(latency, 1),
               "total_detik":   round(time.time() - mulai_total, 1),
               "n_error":       len(daftar_error),
               "jenis_error":   ";".join(daftar_error),
               "token_input":   getattr(usage, "prompt_tokens", None),
               "token_output":  getattr(usage, "completion_tokens", None)})

    if teks:
        CACHE[kunci] = teks
        simpan_cache()
    return teks

# ---------- 5. Lihat prompt satu tanggal (tanpa memanggil LLM) ----------
print(buat_prompt(buat_snapshot(test, TANGGAL_CONTOH["aman"]), TANGGAL_CONTOH["aman"]))

# ---------- 6. Buat ringkasan untuk tiga tanggal contoh ----------
SNAPSHOT  = {nama: buat_snapshot(test, tgl) for nama, tgl in TANGGAL_CONTOH.items()}
RINGKASAN = {}

for nama, tgl in TANGGAL_CONTOH.items():
    RINGKASAN[nama] = ringkas(SNAPSHOT[nama], tgl, coba=6)
    print(f"==================== {nama.upper()}: {format_tanggal(tgl)} ====================")
    print(RINGKASAN[nama])
    print()
    time.sleep(10)
if LANGFUSE_AKTIF:
    get_client().flush()      # kirim semua catatan yang masih tertahan ke Langfuse

Kamu adalah asisten yang menulis laporan stok bahan baku untuk pemilik Tewiwi Bakehouse

TANGGAL:18 Juni 2022
JUMLAH BAHAN PER STATUS:Cukup = 10 bahan
DAFTAR BAHAN PER BAGIAN (sudah final, ditentukan oleh sistem)
**Perlu dibeli:**
Tidak ada

**Berpotensi berlebih:**
Tidak ada

**Perlu dipantau:**
- Telur: sisa 3 butir setelah 7 hari
- Gula: sisa 0,5 kg setelah 7 hari

ARTI ISTILAH :
- kurang: jumlah kekurangan bila tidak memesan lagi dalam 7 hari ke depan.
- sisa ... setelah 7 hari: sisa stok di akhir 7 hari bila tidak memesan lagi.
- Berpotensi berlebih: setelah 7 hari, stok masih tersisa lebih dari 7 hari pemakaian
- Perlu dipantau: stok masih cukup, tetapi sisanya tipis di akhir 7hari.

TULIS LAPORAN DENGAN SUSUNAN BERIKUT:
1. Satu kalimat pembuka yang menyebut jumlah bahan per status persis seperti JUMLAH BAHAN PER STATUS, tanpa menulis label "JUMLAH BAHAN PER STATUS".
2. Ketiga bagian disalin PERSIS seperti DAFTAR BAHAN PER BAGIAN: judul, setiap baris, dan tulisan "Tidak ada".
   

## Cell 6: Deterministic Validation (pemeriksa fakta otomatis)

**Tujuan:** memastikan ringkasan LLM sesuai dengan data dan aturan, **tanpa** memakai LLM lain sebagai penilai.

### Yang diperiksa
| Jenis | Pemeriksaan | Akibat |
|---|---|---|
| Fakta | angka yang tidak ada di data | GAGAL |
| Fakta | judul bagian hilang, bahan terlewat, bahan salah masuk bagian | GAGAL |
| Fakta | **setiap baris harus sama persis dengan Cell 2b** (saran mentor) | GAGAL |
| Fakta | satuan salah (misalnya telur dalam kg) | GAGAL |
| Fakta | saran membeli untuk bahan yang bukan Perlu restok | GAGAL |
| Fakta | kata terlarang (harga, pemasok, cuaca, dan lainnya) | GAGAL |
| Kualitas | penutup hilang, menyalin teks aturan, memuat angka, label prompt ikut tertulis | dicatat saja |

**Kenapa pemeriksaan per baris penting?** Pemeriksaan angka biasa hanya mengecek apakah sebuah angka ada di data. Jika LLM menukar angka Gula dan Cokelat, kedua angka tetap ada di data dan akan lolos. Pemeriksaan per baris mencocokkan bahan, jenis nilai, angka, dan satuan **sekaligus**.

### Uji pemeriksa dengan ringkasan palsu
| Uji | Kasus | Hasil |
|---|---|---|
| 1 | angka karangan, bahan terlewat, kata terlarang | GAGAL (benar) |
| 2 | penutup menyarankan membeli bahan yang hanya perlu dipantau | GAGAL (benar) |
| 3 | telur ditulis dalam kg | GAGAL (benar) |
| 4 | penutup benar (beli bahan kurang, pantau garam) | tidak ada alarm (benar) |
| 5 | angka Gula dan Cokelat ditukar | GAGAL. Pemeriksaan angka lama **tidak** menangkapnya |
| 6 | penutup benar dengan frasa "tunda pembelian" | tidak ada alarm (benar) |

**Keterbatasan:** pemeriksa membaca saran di kalimat penutup dengan aturan kata kerja sederhana. Aturan seperti ini bisa salah membaca susunan kalimat yang tidak biasa (lihat Cell 7b), sehingga pembacaan manual tetap diperlukan.

In [ ]:
# CELL 6 : Pemeriksa fakta otomatis untuk ringkasan GenAI

import re

BAGIAN = {
    "dibeli":   "Perlu dibeli",
    "berlebih": "Berpotensi berlebih",
    "pantau":   "Perlu dipantau",
}
KATA_TERLARANG = ["harga", "rp", "rupiah", "pemasok", "supplier", "merek", "cuaca",
                  "hujan", "musim", "libur", "liburan", "acara", "lebaran", "diskon", "promo"]

# ---------- 1. Pemeriksaan angka ----------
def ke_angka(teks):
    """Ubah teks angka gaya Indonesia (3,8 atau 1.447,9) menjadi float."""
    if "," in teks and "." in teks:
        teks = teks.replace(".", "").replace(",", ".")
    elif "," in teks:
        teks = teks.replace(",", ".")
    elif re.fullmatch(r"\d{1,3}(\.\d{3})+", teks):
        teks = teks.replace(".", "")
    return float(teks)

assert ke_angka("3,8")     == 3.8
assert ke_angka("989,3")   == 989.3
assert ke_angka("1.447,9") == 1447.9
assert ke_angka("1.447")   == 1447.0
assert ke_angka("20")      == 20.0
print("Uji ke_angka: semua benar")

def angka_di_teks(teks):
    # Buang nomor urut daftar ("1. ", "2) ") supaya tidak dianggap angka data
    teks = re.sub(r"(?m)^\s*\d+[.)]\s+", "", teks)
    return [ke_angka(a) for a in re.findall(r"\d+(?:[.,]\d+)*", teks)]

def angka_diizinkan(snap, tanggal):
    nilai = set()
    for kolom in snap.select_dtypes("number").columns:
        nilai |= set(snap[kolom].abs().round(1).tolist())
    nilai |= set(map(float, snap["status"].value_counts().tolist()))   # angka di RINGKASAN JUMLAH
    tgl = pd.Timestamp(tanggal)
    nilai |= {float(tgl.day), float(tgl.year), float(H), 30.0, float(PANTAU_DAYS)}
    return {round(x, 1) for x in nilai}

# ---------- 2. Pemeriksaan bahan per bagian ----------
# Judul dicari yang DIIKUTI TITIK DUA, supaya kalimat pembuka tidak dianggap judul
POLA_JUDUL = {k: re.compile(rf"{re.escape(j)}\s*\**\s*:", re.IGNORECASE) for k, j in BAGIAN.items()}

def potong_bagian(teks):
    posisi = {}
    for kunci, pola in POLA_JUDUL.items():
        cocok = pola.search(teks)
        if cocok:
            posisi[kunci] = cocok.start()

    hasil = {k: "" for k in BAGIAN}
    urut = sorted(posisi.items(), key=lambda x: x[1])
    for i, (kunci, awal) in enumerate(urut):
        akhir = urut[i + 1][1] if i + 1 < len(urut) else len(teks)
        hasil[kunci] = teks[awal:akhir]
    return hasil, [k for k in BAGIAN if k not in posisi]

# Baris daftar: diawali -, *, •, –, —, ATAU angka diikuti titik/kurung
POLA_DAFTAR = re.compile(r"^(?:[-*•–—]|\d+[.)])")

def isi_bagian(teks_bagian):
    """Isi satu bagian: teks setelah judul + baris daftar di bawahnya.
    Berhenti di baris kosong ATAU di baris yang bukan daftar (misalnya kalimat penutup)."""
    if not teks_bagian:
        return ""
    baris = teks_bagian.splitlines()
    setelah_judul = baris[0].split(":", 1)[1].strip(" *") if ":" in baris[0] else ""
    isi = [setelah_judul] if setelah_judul else []
    for b in baris[1:]:
        b = b.strip()
        if b == "":
            if isi:
                break
            continue
        if POLA_DAFTAR.match(b) or b.lower().startswith("tidak ada"):
            isi.append(b)
        else:
            break
    return "\n".join(isi)

def bahan_di_bagian(teks_bagian, daftar_bahan):
    isi = isi_bagian(teks_bagian).lower()
    return {b for b in daftar_bahan if b.lower() in isi}

# ---------- 3. Validasi per baris (saran mentor): bahan + jenis nilai + angka + satuan sekaligus ----------
def rapikan_baris(b):
    """Samakan bentuk baris supaya perbedaan kecil (spasi, tanda tebal, titik di akhir, bullet) tidak dianggap salah."""
    b = b.replace("**", "").strip()
    b = re.sub(r"^[-*•–—]\s*", "- ", b)        # bullet apa pun → "- "
    b = re.sub(r"\s+", " ", b).rstrip(".")
    return b.lower()

def cek_baris(teks, klas):
    """Bandingkan baris daftar di ringkasan dengan baris_laporan dari Cell 2b, per bagian.
    Angka yang tertukar antar bahan, satuan salah, atau jenis nilai salah akan tertangkap di sini."""
    bagian_teks, _ = potong_bagian(teks)
    hilang, asing = [], []
    for kunci, judul in BAGIAN.items():
        harap = {rapikan_baris(b) for b in klas.loc[klas["bagian"] == judul, "baris_laporan"]}
        isi = isi_bagian(bagian_teks[kunci]).splitlines()
        temu = {rapikan_baris(b) for b in isi if POLA_DAFTAR.match(b.strip())}
        hilang += [f"{judul}: {b}" for b in sorted(harap - temu)]
        asing  += [f"{judul}: {b}" for b in sorted(temu - harap)]
    return hilang, asing

# ---------- 4. Pemeriksaan satuan ----------
SATUAN_DIKENAL = r"(kg|kilogram|gram|gr|g|butir|liter|l|ons)"

def cek_satuan(teks, snap):
    """Bahan yang angkanya ditulis dengan satuan berbeda dari kolom satuan (misal Telur 3 kg)."""
    salah = []
    for b, satuan in zip(snap["bahan"], snap["satuan"]):
        pola = rf"{re.escape(b)}[^\n]*?\d+(?:[.,]\d+)*\s*{SATUAN_DIKENAL}\b"
        for m in re.finditer(pola, teks, flags=re.IGNORECASE):
            if m.group(1).lower() != satuan:
                salah.append(b)
                break
    return sorted(set(salah))

# ---------- 5. Pemeriksaan kalimat penutup ----------
def kalimat_penutup(teks):
    """Teks setelah judul bagian TERAKHIR, tanpa baris daftar dan tanpa 'Tidak ada'."""
    akhir_judul = 0
    for pola in POLA_JUDUL.values():
        for m in pola.finditer(teks):
            akhir_judul = max(akhir_judul, m.end())
    sisa = []
    for b in teks[akhir_judul:].splitlines():
        b = b.strip(" *")
        if b == "" or POLA_DAFTAR.match(b) or b.lower().startswith("tidak ada"):
            continue
        sisa.append(b)
    return " ".join(sisa)

KATA_BELI   = ["restok", "membeli", "beli", "memesan", "pesan"]
KATA_PANTAU = ["memantau", "pantau", "awasi", "perhatikan"]
KATA_TAHAN  = ["tunda", "menunda", "tahan", "menahan"]          # kata kerja menunda pembelian
KATA_LARANG = ["jangan", "tunda", "menunda", "tidak perlu", "tidak usah"]

def aksi_penutup(penutup, daftar_bahan):
    """Saran untuk tiap bahan di penutup: 'beli', 'pantau', atau 'tahan',
    dari kata kerja terdekat SEBELUM nama bahan, dalam kalimat yang sama."""
    aksi_bahan = {}
    for kalimat in re.split(r"[.!?]", penutup.lower()):
        kejadian = []
        for kata in KATA_BELI + KATA_PANTAU + KATA_TAHAN:
            for m in re.finditer(rf"\b{kata}\b", kalimat):
                if kata in KATA_BELI:
                    jenis = "beli"
                    sebelum = kalimat[max(0, m.start() - 20):m.start()]
                    if any(l in sebelum for l in KATA_LARANG):   # "jangan membeli", "tunda dulu membeli"
                        jenis = "tahan"
                elif kata in KATA_PANTAU:
                    jenis = "pantau"
                else:
                    jenis = "tahan"                               # "tunda pembelian X", "tahan X"
                kejadian.append((m.start(), "aksi", jenis))
        for b in daftar_bahan:
            for m in re.finditer(re.escape(b.lower()), kalimat):
                kejadian.append((m.start(), "bahan", b))

        aksi_sekarang = None
        for _, tipe, isi in sorted(kejadian):
            if tipe == "aksi":
                aksi_sekarang = isi
            elif aksi_sekarang is not None:
                aksi_bahan[isi] = aksi_sekarang
    return aksi_bahan
# Frasa yang menandakan penutup menyalin teks aturan, bukan saran dengan kata-kata sendiri
FRASA_ATURAN = ["sarankan memesan", "sarankan menunda", "sarankan memantau",
                "bukan membeli atau restok", "di bagian itu", "sesuai aturan",
                "bagian perlu dibeli", "bagian berpotensi berlebih", "bagian perlu dipantau"]

# ---------- 6. Gabungkan semua pemeriksaan ----------
def cek_fakta(teks, snap, tanggal):
    hasil = {}

    # FAKTA: angka
    izin = angka_diizinkan(snap, tanggal)
    hasil["angka_asing"] = sorted({round(a, 1) for a in angka_di_teks(teks)} - izin)

    # FAKTA: bahan per bagian (harapan dihitung sendiri dari status, tidak bergantung pada prompt)
    harapan = {
        "dibeli":   set(snap.loc[snap["status"] == STATUS_ID["Restock"], "bahan"]),
        "berlebih": set(snap.loc[snap["status"] == STATUS_ID["Excess"], "bahan"]),
        "pantau":   set(snap.loc[(snap["status"] == STATUS_ID["Sufficient"])
                                 & (snap["sisa_dalam_hari"] < PANTAU_DAYS), "bahan"]),
    }
    bagian, bagian_hilang = potong_bagian(teks)
    hasil["bagian_hilang"]  = bagian_hilang
    hasil["format_lengkap"] = len(bagian_hilang) == 0
    for kunci, harap in harapan.items():
        ditemukan = bahan_di_bagian(bagian[kunci], snap["bahan"])
        hasil[f"{kunci}_terlewat"]    = sorted(harap - ditemukan)
        hasil[f"{kunci}_salah_masuk"] = sorted(ditemukan - harap)

    # FAKTA: setiap baris harus PERSIS sama dengan hasil klasifikasi rule-based (Cell 2b)
    klas = klasifikasi_rule(snap)
    hasil["baris_hilang"], hasil["baris_asing"] = cek_baris(teks, klas)

    # FAKTA: satuan
    hasil["satuan_salah"] = cek_satuan(teks, snap)

    # FAKTA: saran membeli di penutup hanya boleh untuk bahan Perlu restok
    penutup = kalimat_penutup(teks)
    aksi = aksi_penutup(penutup, snap["bahan"])
    disarankan_beli = {b for b, a in aksi.items() if a == "beli"}
    hasil["penutup_beli_salah"] = sorted(disarankan_beli - harapan["dibeli"])
    sebagian = len(disarankan_beli) > 0 and len(harapan["dibeli"] - disarankan_beli) > 0
    hasil["penutup_beli_sebagian"] = sorted(harapan["dibeli"] - disarankan_beli) if sebagian else []

    # FAKTA: kata terlarang
    hasil["kata_terlarang"] = [k for k in KATA_TERLARANG if re.search(rf"\b{k}\b", teks.lower())]

    # KUALITAS (dicatat, tapi TIDAK membuat GAGAL)
    ada_daftar = (klas["bagian"] != "-").any()
    hasil["kualitas_penutup_hilang"]          = [] if penutup else ["tidak ada kalimat penutup"]
    hasil["kualitas_penutup_menyalin_aturan"] = [f for f in FRASA_ATURAN if f in penutup.lower()]
    hasil["kualitas_penutup_ada_angka"]       = re.findall(r"\d+(?:[.,]\d+)*", re.sub(r"\b7 hari\b", "", penutup))
    hasil["kualitas_penutup_aman_padahal_ada_daftar"] = (
        ["aman"] if ada_daftar and re.search(r"\baman\b", penutup.lower()) else [])

    #kualitas bocor label sistem
    hasil["kualitas_bocor_label_sistem"] = (
        ["label prompt ikut tertulis"] if re.search(r"daftar bahan per bagian|sudah final", teks.lower()) else [])
    # LOLOS = tidak ada kesalahan FAKTA (kolom kualitas_ tidak ikut dihitung)
    hasil["lolos"] = all(len(v) == 0 for k, v in hasil.items()
                         if isinstance(v, list) and not k.startswith("kualitas_"))
    return hasil

# ---------- 7. Periksa ringkasan dari Cell 5 ----------
def tampilkan(judul, hasil):
    print(f"\n{judul}: {'LOLOS' if hasil['lolos'] else 'GAGAL'}")
    for k, v in hasil.items():
        if isinstance(v, list) and v:
            print(f"   {k}: {v}")

print("\n==== Hasil pemeriksaan ringkasan asli ====")
print("Model:", MODEL_GENAI)
for nama, tgl in TANGGAL_CONTOH.items():
    if nama not in RINGKASAN:
        print(f"\n{nama.upper()}: belum ada ringkasan, dilewati")
        continue
    tampilkan(f"{nama.upper()} ({format_tanggal(tgl)})", cek_fakta(RINGKASAN[nama], SNAPSHOT[nama], tgl))

# ---------- 8. Uji pemeriksa dengan ringkasan palsu ----------
# Uji 1: angka karangan, bahan terlewat, bagian hilang, salah masuk, kata terlarang (harus GAGAL)
palsu_1 = """Kondisi stok cukup berat karena musim liburan.

**Perlu dibeli:**
1. Mentega: kurang 99,9 kg
2. Gula: kurang 9,3 kg

**Perlu dipantau:**
- Telur: sisa 3 butir

Harga terigu sedang naik, segera hubungi pemasok."""
tampilkan("==== Uji 1: pemeriksa umum (harus GAGAL) ====",
          cek_fakta(palsu_1, SNAPSHOT["kritis"], TANGGAL_CONTOH["kritis"]))

# Uji 2: penutup menaikkan "dipantau" menjadi "restok" (harus GAGAL: penutup_beli_salah)
palsu_2 = """Ada Cukup = 10 bahan.

**Perlu dibeli:** Tidak ada

**Berpotensi berlebih:** Tidak ada

**Perlu dipantau:**
- Telur: sisa 3 butir setelah 7 hari
- Gula: sisa 0,5 kg setelah 7 hari

Segera lakukan restok untuk Telur dan Gula."""
tampilkan("==== Uji 2: penutup salah (harus GAGAL) ====",
          cek_fakta(palsu_2, SNAPSHOT["aman"], TANGGAL_CONTOH["aman"]))

# Uji 3: telur dalam kg + penutup menyalin aturan (harus GAGAL: satuan_salah ['Telur'])
palsu_3 = """Cukup = 10 bahan.

**Perlu dibeli:** Tidak ada

**Berpotensi berlebih:** Tidak ada

**Perlu dipantau:**
- Telur: sisa 3 kg setelah 7 hari
- Gula: sisa 0,5 kg setelah 7 hari

Untuk bahan Perlu dipantau, sarankan MEMANTAU, BUKAN membeli atau restok."""
tampilkan("==== Uji 3: satuan salah (harus GAGAL) ====",
          cek_fakta(palsu_3, SNAPSHOT["aman"], TANGGAL_CONTOH["aman"]))

# Uji 4: penutup BENAR dari kasus 9 Juni (harus {'Garam': 'pantau'})
penutup_benar = ("Mohon segera membeli bahan yang kurang agar produksi tidak terganggu, "
                 "dan pantau stok garam agar tidak kehabisan.")
hasil_4 = aksi_penutup(penutup_benar, ["Garam", "Gula", "Telur"])
print("\n==== Uji 4: penutup benar ====")
print(hasil_4, "→", "BENAR" if hasil_4 == {"Garam": "pantau"} else "PEMERIKSA SALAH")

# Uji 5: angka BENAR tapi TERTUKAR antar bahan (contoh dari mentor). Harus GAGAL.
if "kritis" in RINGKASAN:
    asli = RINGKASAN["kritis"]
    palsu_5 = (asli.replace("Gula: kurang 9,3 kg", "Gula: kurang TMP")
                   .replace("Cokelat: kurang 7,2 kg", "Cokelat: kurang 9,3 kg")
                   .replace("Gula: kurang TMP", "Gula: kurang 7,2 kg"))
    assert palsu_5 != asli, "Teks tidak berubah, cek penulisan baris Gula/Cokelat di ringkasan kritis"
    hasil_5 = cek_fakta(palsu_5, SNAPSHOT["kritis"], TANGGAL_CONTOH["kritis"])
    tampilkan("==== Uji 5: angka tertukar antar bahan (harus GAGAL) ====", hasil_5)
    print("   angka_asing:", hasil_5["angka_asing"], "← pemeriksaan angka lama TIDAK menangkap pertukaran ini")

# Uji 6: penutup BENAR dengan "tunda pembelian" (kasus 15 Juni). Cream Cheese harus 'tahan', bukan 'beli'.
penutup_tunda = ("Sebaiknya segera pesan Mentega, tunda pembelian Cream Cheese, "
                 "dan pantau terus stok Susu Bubuk serta Tepung Gandum Utuh untuk 7 hari ke depan.")
hasil_6 = aksi_penutup(penutup_tunda, ["Mentega", "Cream Cheese", "Susu Bubuk", "Tepung Gandum Utuh"])
harap_6 = {"Mentega": "beli", "Cream Cheese": "tahan", "Susu Bubuk": "pantau", "Tepung Gandum Utuh": "pantau"}
print("\n==== Uji 6: penutup dengan 'tunda pembelian' ====")
print(hasil_6, "→", "BENAR" if hasil_6 == harap_6 else "PEMERIKSA SALAH")

Uji ke_angka: semua benar

==== Hasil pemeriksaan ringkasan asli ====
Model: openai/gpt-oss-120b

KRITIS (10 Juli 2022): LOLOS
   kualitas_bocor_label_sistem: ['label prompt ikut tertulis']

BERLEBIH (10 September 2022): LOLOS

AMAN (18 Juni 2022): LOLOS
   kualitas_bocor_label_sistem: ['label prompt ikut tertulis']

==== Uji 1: pemeriksa umum (harus GAGAL) ====: GAGAL
   angka_asing: [3.0, 99.9]
   bagian_hilang: ['berlebih']
   dibeli_terlewat: ['Cokelat', 'Cream Cheese', 'Garam', 'Ragi Instan', 'Susu Bubuk', 'Telur', 'Tepung Gandum Utuh', 'Terigu Protein Tinggi']
   pantau_salah_masuk: ['Telur']
   baris_hilang: ['Perlu dibeli: - cokelat: kurang 7,2 kg', 'Perlu dibeli: - cream cheese: kurang 3,8 kg', 'Perlu dibeli: - garam: kurang 6,4 kg', 'Perlu dibeli: - gula: kurang 9,3 kg', 'Perlu dibeli: - mentega: kurang 12,9 kg', 'Perlu dibeli: - ragi instan: kurang 0,9 kg', 'Perlu dibeli: - susu bubuk: kurang 3,8 kg', 'Perlu dibeli: - telur: kurang 58 butir', 'Perlu dibeli: - tepung gandum u

## Cell 6b: Final Inventory Report (guardrail)

**Tujuan:** memastikan owner **tidak pernah** menerima laporan yang gagal validasi.

```
LLM (percobaan 1) → validasi → LOLOS → laporan dari LLM
                             → GAGAL → LLM (jawaban baru) → validasi → LOLOS → laporan dari LLM
                                                                     → GAGAL → laporan TEMPLATE
LLM tidak tersedia ─────────────────────────────────────────────────────────→ laporan TEMPLATE
```

**Laporan template** disusun 100% oleh kode dari hasil Cell 2b. Isinya selalu benar, tetapi narasinya umum dan tidak menyebut nama bahan di kalimat saran.

**Pembersihan:** baris label dari prompt (`DAFTAR BAHAN PER BAGIAN ...`) yang kadang ikut disalin LLM dihapus sebelum divalidasi dan ditampilkan.

**Hasil pengujian:**
- Template lolos validasi di **116 dari 116** tanggal test.
- Saat LLM dibuat selalu salah, sistem menolak dua kali lalu memakai template.
- Saat LLM dibuat tidak tersedia, sistem langsung memakai template.
- Tidak ada laporan final yang memuat label sistem.

In [ ]:
# CELL 6b : Final Inventory Report (guardrail)
#           LLM → validasi → LOLOS? pakai teks LLM
#                          → GAGAL? minta jawaban baru (maks. MAKS_ULANG kali)
#                                   → masih GAGAL / LLM tidak tersedia? pakai laporan TEMPLATE (tanpa LLM)

MAKS_ULANG = 1

# ---------- 1. Laporan template: disusun 100% oleh kode dari hasil Cell 2b ----------
def laporan_template(snap, tanggal):
    klas = klasifikasi_rule(snap)
    jumlah = snap["status"].value_counts()
    pembuka = (f"Per {format_tanggal(tanggal)}, jumlah bahan per status: "
               + ", ".join(f"{status} {n} bahan" for status, n in jumlah.items()) + ".")

    saran = []
    if (klas["bagian"] == "Perlu dibeli").any():
        saran.append("Segera pesan semua bahan yang kurang agar produksi tidak terganggu.")
    if (klas["bagian"] == "Berpotensi berlebih").any():
        saran.append("Tunda dulu pembelian bahan yang stoknya berlebih.")
    if (klas["bagian"] == "Perlu dipantau").any():
        saran.append("Pantau terus pemakaian bahan yang sisanya tipis.")
    if not saran:
        saran.append("Stok aman untuk 7 hari ke depan.")

    return f"{pembuka}\n\n{susun_daftar(klas)}\n\n{' '.join(saran)}"

# Hapus baris label prompt yang kadang ikut disalin model (bukan isi laporan untuk owner)

POLA_LABEL_SISTEM = re.compile(r"(?im)^[ \t]*\**[ \t]*DAFTAR BAHAN PER BAGIAN.*\n?")
def bersihkan_teks(teks):
    return POLA_LABEL_SISTEM.sub("", teks).strip()
# ---------- 2. Laporan final dengan guardrail ----------
def buat_laporan_final(snap, tanggal, pembuat=None, maks_ulang=MAKS_ULANG):
    """pembuat = fungsi pembuat teks (default: ringkas dari Cell 5). Bisa diganti fungsi palsu untuk pengujian."""
    pembuat = pembuat or ringkas
    catatan = []                                   # jejak setiap percobaan, untuk dievaluasi di Cell 7

    for ulang in range(maks_ulang + 1):
        try:
            teks = pembuat(snap, tanggal, coba=6, ulang=ulang)
        except Exception as e:                     # KuotaHabis, server error, dll.
            catatan.append({"ulang": ulang, "lolos": False, "alasan": f"LLM tidak tersedia ({type(e).__name__})"})
            break                                  # LLM sedang tidak bisa dipakai: langsung ke template
        if not teks:
            catatan.append({"ulang": ulang, "lolos": False, "alasan": "jawaban kosong"})
            continue

        teks = bersihkan_teks(teks)                # ← buang label sistem SEBELUM divalidasi dan ditampilkan

        hasil = cek_fakta(teks, snap, tanggal)
        masalah = [k for k, v in hasil.items()
                   if isinstance(v, list) and v and not k.startswith("kualitas_")]
        catatan.append({"ulang": ulang, "lolos": hasil["lolos"], "alasan": ", ".join(masalah)})
        if hasil["lolos"]:
            return {"teks": teks, "sumber": "llm" if ulang == 0 else "llm_ulang", "catatan": catatan}

    return {"teks": laporan_template(snap, tanggal), "sumber": "template", "catatan": catatan}

# ---------- 3. Uji: laporan template harus SELALU lolos validasi ----------
print("==== Uji template di semua tanggal test ====")
gagal_template = []
for tgl in pd.to_datetime(test["date"].unique()):
    s = buat_snapshot(test, tgl)
    if not cek_fakta(laporan_template(s, tgl), s, tgl)["lolos"]:
        gagal_template.append(tgl.date())
print(f"Template lolos validasi: {len(test['date'].unique()) - len(gagal_template)} dari {len(test['date'].unique())} tanggal")
print("Tanggal yang gagal:", gagal_template if gagal_template else "tidak ada")

# ---------- 4. Uji guardrail dengan pembuat teks PALSU (tidak memakai kuota LLM) ----------
def pembuat_selalu_salah(snap, tanggal, coba=6, ulang=0):
    """Selalu menukar angka Gula dan Cokelat (kasus mentor), supaya validasi selalu menolak."""
    return palsu_5

def pembuat_tidak_tersedia(snap, tanggal, coba=6, ulang=0):
    raise KuotaHabis("simulasi server penuh")

for judul, fungsi in [("LLM selalu salah", pembuat_selalu_salah), ("LLM tidak tersedia", pembuat_tidak_tersedia)]:
    final = buat_laporan_final(SNAPSHOT["kritis"], TANGGAL_CONTOH["kritis"], pembuat=fungsi)
    print(f"\n==== Uji guardrail: {judul} ====")
    print("Sumber laporan:", final["sumber"], "(harus: template)")
    for c in final["catatan"]:
        print("  percobaan", c)

# ---------- 5. Laporan final untuk tiga tanggal contoh (memakai LLM asli, dari cache) ----------
LAPORAN_FINAL = {}
for nama, tgl in TANGGAL_CONTOH.items():
    LAPORAN_FINAL[nama] = buat_laporan_final(SNAPSHOT[nama], tgl)
    print(f"\n==================== FINAL {nama.upper()} | sumber: {LAPORAN_FINAL[nama]['sumber']} ====================")
    print(LAPORAN_FINAL[nama]["teks"])

print("\n==== Contoh laporan TEMPLATE (tanggal kritis) ====")
print(laporan_template(SNAPSHOT["kritis"], TANGGAL_CONTOH["kritis"]))

# Uji: laporan final yang ditampilkan ke owner TIDAK boleh memuat label sistem
bocor_final = [nama for nama, lap in LAPORAN_FINAL.items()
               if re.search(r"daftar bahan per bagian|sudah final", lap["teks"].lower())]
print("\nLaporan final yang masih memuat label sistem:", bocor_final if bocor_final else "tidak ada ✅")

==== Uji template di semua tanggal test ====
Template lolos validasi: 116 dari 116 tanggal
Tanggal yang gagal: tidak ada

==== Uji guardrail: LLM selalu salah ====
Sumber laporan: template (harus: template)
  percobaan {'ulang': 0, 'lolos': False, 'alasan': 'baris_hilang, baris_asing'}
  percobaan {'ulang': 1, 'lolos': False, 'alasan': 'baris_hilang, baris_asing'}

==== Uji guardrail: LLM tidak tersedia ====
Sumber laporan: template (harus: template)
  percobaan {'ulang': 0, 'lolos': False, 'alasan': 'LLM tidak tersedia (KuotaHabis)'}

==================== FINAL KRITIS | sumber: llm ====================
Perlu restok = 10 bahan.

**Perlu dibeli:**
- Susu Bubuk: kurang 3,8 kg
- Mentega: kurang 12,9 kg
- Ragi Instan: kurang 0,9 kg
- Cokelat: kurang 7,2 kg
- Garam: kurang 6,4 kg
- Terigu Protein Tinggi: kurang 257,2 kg
- Gula: kurang 9,3 kg
- Telur: kurang 58 butir
- Cream Cheese: kurang 3,8 kg
- Tepung Gandum Utuh: kurang 6,1 kg

**Berpotensi berlebih:**
Tidak ada

**Perlu dipantau:**
Tid

## Cell 7: Evaluasi di 30 tanggal acak

**Sampel:** 30 tanggal dipilih acak (seed 42) dari periode test, **tanpa** tiga tanggal contoh. Sampel mencakup 18 tanggal dengan bahan Perlu restok, 7 dengan bahan Berlebih, dan 15 dengan bahan Perlu dipantau.

**Dua angka yang dilaporkan berdampingan:**
1. **Kepatuhan LLM pada percobaan pertama**, yaitu seberapa sering LLM benar tanpa bantuan.
2. **Kepatuhan laporan final setelah guardrail**, yaitu yang benar-benar dilihat owner.

**Hasil (Groq gpt-oss-120b, prompt v3):**

| Metrik | Hasil |
|---|---|
| Lolos validasi pada percobaan pertama | **30 dari 30** |
| Laporan final lolos validasi | **30 dari 30** |
| Catatan kualitas | label prompt ikut tertulis (2), penutup mirip teks aturan (1) |
| Latency median / p90 | 1,9 / 2,5 detik |
| Error rate per percobaan | 0,8% (1 error, berasal dari bentrok versi library Langfuse, bukan dari Groq) |
| Token rata-rata | sekitar 700 masuk, 760 keluar |

**Catatan kejujuran:**
- Dibanding uji awal (GLM-4.7-Flash dengan prompt v1, sekitar 70 sampai 80% lolos), ada **dua perubahan sekaligus**: prompt (v1 ke v3) dan model (GLM ke gpt-oss-120b). Kenaikan hasil tidak bisa dikaitkan hanya ke salah satunya.
- "Lolos" berarti lolos pemeriksa otomatis. Pembacaan manual menemukan kebocoran label prompt yang tidak dirancang untuk ditangkap pemeriksa fakta, dan kini ditangani di Cell 6b.
- 30 tanggal adalah sampel. Cakupan penuh 116 tanggal ada di Cell 8.

In [ ]:
# CELL 7 : Evaluasi di 30 tanggal acak periode test
#          (1) kepatuhan LLM pada percobaan PERTAMA  → jawaban jujur untuk RQ3
#          (2) kepatuhan LAPORAN FINAL setelah guardrail (Cell 6b) → yang benar-benar dilihat owner
#          (3) observability: latency dan error dari log_llm.csv

# 1. Tanggal contoh (Cell 3) dipakai untuk menyetel prompt, jadi DIKELUARKAN dari evaluasi
tanggal_contoh = set(pd.to_datetime(list(TANGGAL_CONTOH.values())))
semua_tanggal = np.sort([t for t in pd.to_datetime(test["date"].unique()) if t not in tanggal_contoh])

N_SAMPEL = 30
rng = np.random.default_rng(SEED)
sampel_tanggal = sorted(rng.choice(semua_tanggal, size=N_SAMPEL, replace=False))
print(f"Tanggal tersedia (tanpa tanggal contoh): {len(semua_tanggal)} | Sampel: {N_SAMPEL} | "
      f"Model: {MODEL_GENAI} | Prompt: {PROMPT_VERSI}")

# 2. Buat, periksa, dan susun laporan final. Kalau LLM sedang tidak tersedia, proses BERHENTI
#    (bukan dicatat sebagai GAGAL). Jalankan ulang nanti: yang sudah jadi diambil dari cache.
baris = []
for i, tgl in enumerate(sampel_tanggal, start=1):
    snap = buat_snapshot(test, tgl)

    try:
        teks_1 = ringkas(snap, tgl, coba=6, ulang=0)          # percobaan pertama
    except Exception as e:
        print(f"\nLLM tidak tersedia di tanggal ke-{i} ({type(e).__name__}). "
              f"{i - 1} tanggal sudah selesai. Jalankan ulang cell ini nanti.")
        break
    if not teks_1:
        print(f"\nLLM mengembalikan jawaban kosong di tanggal ke-{i}. Proses dihentikan untuk dicek.")
        break

    hasil_1 = cek_fakta(teks_1, snap, tgl)
    final   = buat_laporan_final(snap, tgl)                   # ulang=0 diambil dari cache
    if any(c["alasan"].startswith("LLM tidak tersedia") for c in final["catatan"]):
        print(f"\nLLM tidak tersedia saat percobaan ulang di tanggal ke-{i}. "
              f"{i - 1} tanggal sudah selesai. Jalankan ulang cell ini nanti.")
        break

    baris.append({
        "date":         pd.Timestamp(tgl),
        "n_restok":     int((snap["status"] == STATUS_ID["Restock"]).sum()),
        "n_berlebih":   int((snap["status"] == STATUS_ID["Excess"]).sum()),
        "n_pantau":     int(((snap["status"] == STATUS_ID["Sufficient"])
                             & (snap["sisa_dalam_hari"] < PANTAU_DAYS)).sum()),
        "ringkasan":    teks_1,
        **{k: (len(v) if isinstance(v, list) else v) for k, v in hasil_1.items()},
        "sumber_final": final["sumber"],
        "n_percobaan":  len(final["catatan"]),
        "final_lolos":  cek_fakta(final["teks"], snap, tgl)["lolos"],
        "teks_final":   final["teks"],
    })
    print(f"{i:2d}/{N_SAMPEL}  {pd.Timestamp(tgl).date()}  →  percobaan 1: "
          f"{'LOLOS' if hasil_1['lolos'] else 'GAGAL'} | final: {final['sumber']}")
    time.sleep(10)

eval_genai = pd.DataFrame(baris)
print(f"\nSelesai diperiksa: {len(eval_genai)} dari {N_SAMPEL} tanggal"
      + ("" if len(eval_genai) == N_SAMPEL else "  (BELUM LENGKAP, jalankan ulang cell ini nanti)"))

# 3. Ringkasan hasil
if len(eval_genai) > 0:
    kolom_info     = ["date", "n_restok", "n_berlebih", "n_pantau", "ringkasan", "lolos", "format_lengkap",
                      "sumber_final", "n_percobaan", "final_lolos", "teks_final"]
    kolom_kualitas = [c for c in eval_genai.columns if c.startswith("kualitas_")]
    kolom_fakta    = [c for c in eval_genai.columns if c not in kolom_info + kolom_kualitas]

    print(f"\n==== (1) Kepatuhan LLM, percobaan pertama: {eval_genai['lolos'].mean() * 100:.1f}% "
          f"({eval_genai['lolos'].sum()} dari {len(eval_genai)}) ====")
    print("Total kesalahan fakta per jenis (hanya yang muncul):")
    total = eval_genai[kolom_fakta].sum()
    print(total[total > 0].to_string() if (total > 0).any() else "  tidak ada")

    print("\nCatatan kualitas per jenis (tidak membuat GAGAL):")
    total_k = eval_genai[kolom_kualitas].sum()
    print(total_k[total_k > 0].to_string() if (total_k > 0).any() else "  tidak ada")

    print(f"\n==== (2) Laporan final setelah guardrail: {eval_genai['final_lolos'].mean() * 100:.1f}% lolos validasi ====")
    print("Sumber laporan final:", eval_genai["sumber_final"].value_counts().to_dict())

    print("\n==== Cakupan kondisi di sampel ====")
    print("Tanggal dengan bahan Perlu restok  :", (eval_genai["n_restok"] > 0).sum())
    print("Tanggal dengan bahan Berlebih      :", (eval_genai["n_berlebih"] > 0).sum())
    print("Tanggal dengan bahan Perlu dipantau:", (eval_genai["n_pantau"] > 0).sum())

    # Teks yang GAGAL di percobaan pertama, untuk dibaca manual (kesalahan sungguhan atau alarm palsu?)
    gagal = eval_genai[~eval_genai["lolos"]]
    print(f"\n==== Percobaan pertama yang GAGAL: {len(gagal)} ====")
    for _, r in gagal.iterrows():
        masalah = [c for c in kolom_fakta if r[c] > 0]
        print("=" * 70)
        print(r["date"].date(), "| masalah:", masalah, "| final:", r["sumber_final"])
        print("-" * 70)
        print(r["ringkasan"])

    eval_genai.to_csv(OUTPUT_DIR / f"evaluasi_genai_{PROMPT_VERSI}.csv", index=False)
    print("\nTersimpan:", OUTPUT_DIR / f"evaluasi_genai_{PROMPT_VERSI}.csv")

# 4. Observability: berapa lama dan seberapa sering error (dari log_llm.csv)
if LOG_FILE.exists():
    log = pd.read_csv(LOG_FILE)
    log = log[(log["versi_prompt"] == PROMPT_VERSI) & (log["model"] == MODEL_GENAI)]
    sukses = log[log["status"] == "sukses"]
    n_error = log["n_error"].sum()

    print(f"\n==== (3) Observability LLM (prompt {PROMPT_VERSI}, model {MODEL_GENAI}) ====")
    print(f"Panggilan tercatat      : {len(log)} (sukses {len(sukses)}, gagal {(log['status'] == 'gagal').sum()})")
    if len(sukses) > 0:
        print(f"Latency jawaban (detik) : median {sukses['latency_detik'].median():.1f} | "
              f"p90 {sukses['latency_detik'].quantile(0.9):.1f} | maks {sukses['latency_detik'].max():.1f}")
        print(f"Total per panggilan     : median {sukses['total_detik'].median():.1f} detik (termasuk menunggu retry)")
        print(f"Token rata-rata         : input {sukses['token_input'].mean():.0f} | output {sukses['token_output'].mean():.0f}")
    if n_error + len(sukses) > 0:
        print(f"Error rate per percobaan: {n_error / (n_error + len(sukses)) * 100:.1f}% ({n_error} error)")
    print("Jenis error:", log["jenis_error"].dropna().str.split(";").explode().value_counts().to_dict())
    if LANGFUSE_AKTIF:
      get_client().flush()      # kirim semua catatan yang masih tertahan ke Langfuse

Tanggal tersedia (tanpa tanggal contoh): 113 | Sampel: 30 | Model: openai/gpt-oss-120b | Prompt: v3
 1/30  2022-06-08  →  percobaan 1: LOLOS | final: llm
 2/30  2022-06-09  →  percobaan 1: LOLOS | final: llm
 3/30  2022-06-13  →  percobaan 1: LOLOS | final: llm
 4/30  2022-06-20  →  percobaan 1: LOLOS | final: llm
 5/30  2022-06-21  →  percobaan 1: LOLOS | final: llm
 6/30  2022-07-11  →  percobaan 1: LOLOS | final: llm
 7/30  2022-07-17  →  percobaan 1: LOLOS | final: llm
 8/30  2022-07-19  →  percobaan 1: LOLOS | final: llm
 9/30  2022-07-22  →  percobaan 1: LOLOS | final: llm
10/30  2022-07-23  →  percobaan 1: LOLOS | final: llm
11/30  2022-07-24  →  percobaan 1: LOLOS | final: llm
12/30  2022-07-25  →  percobaan 1: LOLOS | final: llm
13/30  2022-07-29  →  percobaan 1: LOLOS | final: llm
14/30  2022-08-03  →  percobaan 1: LOLOS | final: llm
15/30  2022-08-05  →  percobaan 1: LOLOS | final: llm
16/30  2022-08-07  →  percobaan 1: LOLOS | final: llm
17/30  2022-08-12  →  percobaan 1: L

## Cell 8: Menyimpan data untuk aplikasi Streamlit

**Tujuan:** membuat laporan final untuk **seluruh 116 tanggal test**, lalu menyimpannya agar aplikasi cukup membaca file.

| File | Isi |
|---|---|
| `snapshot_test.csv` | kondisi stok dan hasil klasifikasi 10 bahan × 116 tanggal |
| `laporan_final.csv` | laporan final per tanggal beserta sumbernya (llm, llm_ulang, template) |
| `ringkasan_sistem.json` | metrik model, hasil evaluasi GenAI, dan angka observability |

**Hasil:** 116 dari 116 laporan memakai narasi LLM dan semuanya lolos validasi.

**Keputusan desain aplikasi:** aplikasi Streamlit **hanya membaca laporan yang sudah tersimpan** dan tidak memanggil LLM secara langsung. Alasannya:
- aplikasi tetap berjalan walaupun layanan LLM sedang penuh atau berubah,
- tidak ada API key yang perlu disimpan di server aplikasi,
- laporan yang tampil selalu sudah lolos validasi.

Pola ini mirip laporan terjadwal: laporan dibuat setiap pagi lalu dibaca owner kapan saja.

In [ ]:
# CELL 8 : Simpan laporan final SEMUA tanggal test untuk aplikasi Streamlit (opsi A: aplikasi hanya membaca file)
#          Yang sudah ada di cache tidak memanggil LLM lagi. Kalau LLM sedang tidak tersedia, proses berhenti rapi:
#          jalankan ulang cell ini nanti, tanggal yang sudah jadi diambil dari cache.

APP_DIR = OUTPUT_DIR / "app_data"
APP_DIR.mkdir(parents=True, exist_ok=True)

semua_tgl = sorted(pd.to_datetime(test["date"].unique()))
print(f"Jumlah tanggal test: {len(semua_tgl)} | Model: {MODEL_GENAI} | Prompt: {PROMPT_VERSI}")

# ---------- 1. Snapshot + hasil klasifikasi untuk semua tanggal (tanpa LLM) ----------
snapshot_semua = []
for tgl in semua_tgl:
    k = klasifikasi_rule(buat_snapshot(test, tgl)).drop(columns="date", errors="ignore")
    k.insert(0, "date", tgl.date())
    snapshot_semua.append(k)
snapshot_semua = pd.concat(snapshot_semua, ignore_index=True)
snapshot_semua.to_csv(APP_DIR / "snapshot_test.csv", index=False)
print("Tersimpan: snapshot_test.csv,", len(snapshot_semua), "baris")

# ---------- 2. Laporan final per tanggal (LLM → validasi → guardrail) ----------
laporan = []
for i, tgl in enumerate(semua_tgl, start=1):
    snap = buat_snapshot(test, tgl)
    jumlah_cache = len(CACHE)
    final = buat_laporan_final(snap, tgl)

    if any(c["alasan"].startswith("LLM tidak tersedia") for c in final["catatan"]):
        print(f"\nLLM tidak tersedia di tanggal ke-{i} ({tgl.date()}). {i - 1} tanggal sudah selesai. "
              f"Jalankan ulang cell ini nanti.")
        break

    laporan.append({
        "date":        tgl.date(),
        "sumber":      final["sumber"],
        "n_percobaan": len(final["catatan"]),
        "lolos":       cek_fakta(final["teks"], snap, tgl)["lolos"],
        "laporan":     final["teks"],
    })
    if len(CACHE) > jumlah_cache:            # ada panggilan baru ke LLM → beri jeda
        print(f"{i:3d}/{len(semua_tgl)}  {tgl.date()}  → {final['sumber']} (baru)")
        time.sleep(10)

laporan = pd.DataFrame(laporan)
laporan.to_csv(APP_DIR / "laporan_final.csv", index=False)
lengkap = len(laporan) == len(semua_tgl)
print(f"\nTersimpan: laporan_final.csv, {len(laporan)} dari {len(semua_tgl)} tanggal"
      + ("" if lengkap else "  (BELUM LENGKAP, jalankan ulang cell ini nanti)"))
if len(laporan) > 0:
    print("Sumber laporan:", laporan["sumber"].value_counts().to_dict())
    print("Semua laporan lolos validasi:", bool(laporan["lolos"].all()))

# ---------- 3. Angka ringkas untuk halaman "Tentang sistem" ----------
ev = pd.read_csv(OUTPUT_DIR / f"evaluasi_genai_{PROMPT_VERSI}.csv")
log_llm = pd.read_csv(LOG_FILE)
log_llm = log_llm[(log_llm["versi_prompt"] == PROMPT_VERSI) & (log_llm["model"] == MODEL_GENAI)]
sukses = log_llm[log_llm["status"] == "sukses"]
n_err = int(log_llm["n_error"].sum())

ringkasan_sistem = {
    "dibuat_pada": pd.Timestamp.now(tz="Asia/Jakarta").strftime("%Y-%m-%d %H:%M"),
    "periode_test": [str(semua_tgl[0].date()), str(semua_tgl[-1].date())],
    "model_ml": {
        "nama": "HistGradientBoostingRegressor (quantile 0.7) + inventory policy",
        "metrik_test": bundle["metrik_test"],
    },
    "genai": {
        "model": MODEL_GENAI,
        "versi_prompt": PROMPT_VERSI,
        "n_sampel_evaluasi": int(len(ev)),
        "kepatuhan_percobaan_pertama": round(float(ev["lolos"].mean()), 3),
        "laporan_final_lolos": round(float(ev["final_lolos"].mean()), 3),
        "catatan_kualitas": {c: int(ev[c].sum()) for c in ev.columns if c.startswith("kualitas_")},
    },
    "observability": {
        "panggilan": int(len(log_llm)),
        "latency_median_detik": round(float(sukses["latency_detik"].median()), 1) if len(sukses) else None,
        "latency_p90_detik": round(float(sukses["latency_detik"].quantile(0.9)), 1) if len(sukses) else None,
        "error_rate": round(n_err / (n_err + len(sukses)), 3) if (n_err + len(sukses)) else None,
        "token_input_rata2": round(float(sukses["token_input"].mean())) if len(sukses) else None,
        "token_output_rata2": round(float(sukses["token_output"].mean())) if len(sukses) else None,
    },
}
(APP_DIR / "ringkasan_sistem.json").write_text(json.dumps(ringkasan_sistem, indent=2, ensure_ascii=False, default=str))
print("\nTersimpan: ringkasan_sistem.json")
print(json.dumps(ringkasan_sistem, indent=2, ensure_ascii=False, default=str))

if LANGFUSE_AKTIF:
    get_client().flush()

Jumlah tanggal test: 116 | Model: openai/gpt-oss-120b | Prompt: v3
Tersimpan: snapshot_test.csv, 1160 baris

Tersimpan: laporan_final.csv, 116 dari 116 tanggal
Sumber laporan: {'llm': 116}
Semua laporan lolos validasi: True

Tersimpan: ringkasan_sistem.json
{
  "dibuat_pada": "2026-11-12 00:25",
  "periode_test": [
    "2022-06-01",
    "2022-09-24"
  ],
  "model_ml": {
    "nama": "HistGradientBoostingRegressor (quantile 0.7) + inventory policy",
    "metrik_test": {
      "Baseline (rata-rata x 7)": {
        "meleset_hari": 2.6522484659713617,
        "recall_restock": 0.5157894736842106,
        "f1_restock": 0.5616045845272206,
        "f1_excess": 0.0,
        "f1_macro": 0.4335499040205189
      },
      "Baseline + margin 5%": {
        "meleset_hari": 2.686510267463454,
        "recall_restock": 0.5736842105263158,
        "f1_restock": 0.5883940620782726,
        "f1_excess": 0.0,
        "f1_macro": 0.4410384505114278
      },
      "GB quantile 0.7": {
        "meleset_hari

In [ ]:
for t in ["2022-06-15", "2022-06-17"]:
    tgl = pd.Timestamp(t)
    snap = buat_snapshot(test, tgl)
    print("#" * 70, "\n", t)
    display(klasifikasi_rule(snap)[["bahan", "status", "bagian", "baris_laporan"]])
    for ulang in [0, 1]:
        teks = bersihkan_teks(ringkas(snap, tgl, ulang=ulang))
        hasil = cek_fakta(teks, snap, tgl)
        masalah = {k: v for k, v in hasil.items()
                   if isinstance(v, list) and v and not k.startswith("kualitas_")}
        print(f"\n--- percobaan ulang={ulang} | masalah: {masalah}")
        print(teks)

###################################################################### 
 2022-06-15


,bahan,status,bagian,baris_laporan
0,Mentega,Perlu restok,Perlu dibeli,"- Mentega: kurang 0,8 kg"
1,Cream Cheese,Berlebih,Berpotensi berlebih,"- Cream Cheese: sisa 11,2 kg setelah 7 hari"
2,Susu Bubuk,Cukup,Perlu dipantau,"- Susu Bubuk: sisa 0,2 kg setelah 7 hari"
3,Tepung Gandum Utuh,Cukup,Perlu dipantau,"- Tepung Gandum Utuh: sisa 10,8 kg setelah 7 hari"
4,Gula,Cukup,-,
5,Telur,Cukup,-,
6,Garam,Cukup,-,
7,Terigu Protein Tinggi,Cukup,-,
8,Cokelat,Cukup,-,
9,Ragi Instan,Cukup,-,



--- percobaan ulang=0 | masalah: {'penutup_beli_salah': ['Cream Cheese']}
Pada tanggal 15 Juni 2022, terdapat Cukup = 8 bahan, Perlu restok = 1 bahan, Berlebih = 1 bahan.
**Perlu dibeli:**
- Mentega: kurang 0,8 kg

**Berpotensi berlebih:**
- Cream Cheese: sisa 11,2 kg setelah 7 hari

**Perlu dipantau:**
- Susu Bubuk: sisa 0,2 kg setelah 7 hari
- Tepung Gandum Utuh: sisa 10,8 kg setelah 7 hari

Sebaiknya segera pesan Mentega, tunda pembelian Cream Cheese, dan pantau terus stok Susu Bubuk serta Tepung Gandum Utuh untuk 7 hari ke depan.

--- percobaan ulang=1 | masalah: {'penutup_beli_salah': ['Cream Cheese']}
Pada tanggal 15 Juni 2022, terdapat Cukup = 8 bahan, Perlu restok = 1 bahan, Berlebih = 1 bahan.

**Perlu dibeli:**
- Mentega: kurang 0,8 kg

**Berpotensi berlebih:**
- Cream Cheese: sisa 11,2 kg setelah 7 hari

**Perlu dipantau:**
- Susu Bubuk: sisa 0,2 kg setelah 7 hari
- Tepung Gandum Utuh: sisa 10,8 kg setelah 7 hari

Sebaiknya pesan Mentega sekarang, tunda pembelian Cream Chee

,bahan,status,bagian,baris_laporan
0,Tepung Gandum Utuh,Perlu restok,Perlu dibeli,"- Tepung Gandum Utuh: kurang 6,3 kg"
1,Susu Bubuk,Berlebih,Berpotensi berlebih,"- Susu Bubuk: sisa 4,2 kg setelah 7 hari"
2,Gula,Cukup,Perlu dipantau,"- Gula: sisa 1,7 kg setelah 7 hari"
3,Telur,Cukup,Perlu dipantau,- Telur: sisa 13 butir setelah 7 hari
4,Terigu Protein Tinggi,Cukup,-,
5,Garam,Cukup,-,
6,Cokelat,Cukup,-,
7,Ragi Instan,Cukup,-,
8,Mentega,Cukup,-,
9,Cream Cheese,Cukup,-,



--- percobaan ulang=0 | masalah: {'penutup_beli_salah': ['Susu Bubuk']}
Pada tanggal 17 Juni 2022, terdapat Cukup = 8 bahan, Perlu restok = 1 bahan, Berlebih = 1 bahan.

**Perlu dibeli:**
- Tepung Gandum Utuh: kurang 6,3 kg

**Berpotensi berlebih:**
- Susu Bubuk: sisa 4,2 kg setelah 7 hari

**Perlu dipantau:**
- Gula: sisa 1,7 kg setelah 7 hari
- Telur: sisa 13 butir setelah 7 hari

Sebaiknya pesan Tepung Gandum Utuh sekarang, tunda pembelian Susu Bubuk, dan terus pantau Gula serta Telur selama 7 hari ke depan.

--- percobaan ulang=1 | masalah: {'penutup_beli_salah': ['Susu Bubuk']}
Pada tanggal 17 Juni 2022, terdapat Cukup = 8 bahan, Perlu restok = 1 bahan, Berlebih = 1 bahan.

**Perlu dibeli:**
- Tepung Gandum Utuh: kurang 6,3 kg

**Berpotensi berlebih:**
- Susu Bubuk: sisa 4,2 kg setelah 7 hari

**Perlu dipantau:**
- Gula: sisa 1,7 kg setelah 7 hari
- Telur: sisa 13 butir setelah 7 hari

Sebaiknya segera pesan Tepung Gandum Utuh, tunda pembelian Susu Bubuk, dan pantau terus stok Gu

## Kesimpulan RQ3

1. GenAI dapat menghasilkan ringkasan kondisi stok yang **hanya memuat informasi dari hasil klasifikasi**, dengan syarat keputusan angka dan kategori dibuat oleh kode, bukan oleh LLM.
2. Dengan desain berlapis (rule-based classification, prompt v3, deterministic validation, guardrail), **116 dari 116** laporan test lolos validasi dan memakai narasi LLM.
3. Guardrail menjamin owner tidak menerima laporan yang gagal validasi. Jika LLM salah atau tidak tersedia, laporan template yang tetap benar ditampilkan.

## Temuan dari pembacaan manual
| Temuan | Penanganan |
|---|---|
| Alarm palsu: saran "beli bahan kurang, pantau garam" dianggap salah | pemeriksa membaca saran per bahan |
| Label prompt ikut tertulis di 2 dari 30 ringkasan | dibersihkan di Cell 6b |
| Alarm palsu: frasa "tunda pembelian" tidak dikenali | kata kerja menunda ditambahkan, Uji 6 |

Evaluasi otomatis tidak cukup sendirian. Ketiga temuan ini hanya terlihat lewat pembacaan manual.

## Keterbatasan
- Data stok adalah **data simulasi**, bukan data asli Tewiwi.
- Pemeriksa saran berbasis aturan kata kerja, sehingga bisa salah membaca kalimat dengan susunan tidak biasa. Kesalahannya dirancang agar jatuh ke arah aman (template).
- Pemeriksa belum menilai kalimat pembuka dan alasan secara mendalam.
- Hasil bergantung pada layanan LLM gratis yang bisa berubah atau dihentikan sewaktu-waktu.
- Prompt v3 punya kelemahan kecil (label ikut tersalin), yang ditangani di lapisan validasi, bukan dengan mengubah prompt yang sudah dibekukan.

## Rencana pengembangan
- Tombol "buat ulang laporan dengan LLM" di aplikasi (opsi B).
- RAG untuk menambahkan konteks resep atau SOP Tewiwi ke dalam penjelasan.
- Menjalankan LLM open-weight (misalnya Gemma) secara mandiri untuk mengurangi ketergantungan pada API pihak luar.
- Manajemen versi prompt di Langfuse.